## Bronze ingestion - Auto Loader (production)

In [0]:
from pyspark.sql import functions as F

catalog = "dbr_dev_ua5816bd"
login = "lena066636"

raw_path = f"abfss://{login}@dlsua5816bd.dfs.core.windows.net/raw_streaming/"
checkpoint_path = f"abfss://{login}@dlsua5816bd.dfs.core.windows.net/_checkpoints/bronze_streaming"
schema_path = checkpoint_path + "/schema"
bronze_table = f"{catalog}.{login}_bronze.orders_streaming"


In [0]:
df_raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(raw_path))


In [0]:
df_bronze = (df_raw
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date()))


In [0]:
(df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table))
